# VectoVecto — Tier B Training (DRUNet + VGG + PatchGAN on DIV2K)

This notebook trains the **Deep Unfolding SR** model (DRUNet denoiser + VGG19 perceptual loss + PatchGAN adversarial loss) on **DIV2K** using **Real-ESRGAN second-order degradation** — the Tier B pipeline from the [VectoVecto repo](https://github.com/DontHash/VectoVecto).

## Setup checklist (do this before running)
1. **Accelerator**: `Settings → Accelerator → GPU T4 x2` (this notebook auto-fails if GPU is off)
2. **Internet**: `Settings → Internet → On` (needed to clone the repo + install packages + auto-download VGG19)
3. **DIV2K input**: `Add Input → Datasets → search "div2k" → add`. The notebook auto-detects it at `/kaggle/input/div2k-dataset/...`
4. **Run All**.

**Output**: `best_checkpoint.pth` lands in `/kaggle/working/deep_sr/`. Download it from the `Output` panel on the right, then drop into `artifacts/deep_sr/` locally — `inference.py` auto-uses it.

**Time**: ~24-48h for the full 100 epochs (25k iters) on T4. Reduce `--epochs` below if you want a quick checkpoint sooner.

In [ ]:
# Cell 1 — Clone the repo + install deps (Internet must be ON)
import subprocess, sys, os

REPO_URL = 'https://github.com/DontHash/VectoVecto.git'
REPO_DIR = '/kaggle/working/VectoVecto'

if not os.path.isdir(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Repo already at {REPO_DIR}')

os.chdir(REPO_DIR)
print('Repo contents:')
for f in sorted(os.listdir('.')):
    print(' ', f)

print('\nInstalling deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=False)
print('Done.')

In [ ]:
# Cell 2 — Verify GPU + environment (auto-fails if GPU not enabled)
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU not enabled! Settings → Accelerator → GPU T4 x2, then re-run.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
free, total = torch.cuda.mem_get_info(0)
print(f'VRAM: free {free/1e9:.2f} GB / total {total/1e9:.2f} GB')

In [ ]:
# Cell 3 — Locate the DIV2K dataset (must be added as Input)
import os

div2k_root = '/kaggle/input'
found_dir = None
if os.path.isdir(div2k_root):
    for entry in os.listdir(div2k_root):
        base = os.path.join(div2k_root, entry)
        if os.path.isdir(base):
            for sub in ('DIV2K_train_HR', 'DIV2K_train_HR/DIV2K_train_HR', 'train_HR'):
                p = os.path.join(base, sub)
                if os.path.isdir(p):
                    imgs = [f for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                    if imgs:
                        found_dir = p
                        break
        if found_dir: break

if found_dir is None:
    raise RuntimeError('DIV2K not found. Add Input → Datasets → search "div2k" → Add.')
print(f'DIV2K HR dir: {found_dir}')
print(f'HR images: {len(os.listdir(found_dir))}')
os.environ['DIV2K_HR_DIR'] = found_dir

# Point checkpoints to /kaggle/working/ so Kaggle saves them on exit
out_dir = '/kaggle/working/deep_sr'
os.makedirs(out_dir, exist_ok=True)
os.environ['ARTIFACTS_DIR'] = out_dir
print(f'Checkpoints → {out_dir}')

In [ ]:
# Cell 4 — Smoke test (1 forward+backward step, synthetic data, no DIV2K needed)
# Verifies all components compose: DRUNet + DeepUnfoldingSR + degradation + VGG + PatchGAN
# First run downloads VGG19 (~548MB) for perceptual loss — takes a couple minutes
!python train_deep_sr.py --smoke --device cuda

In [ ]:
# Cell 5 — LAUNCH TRAINING
# T4-tuned defaults: patch=32, batch=8, 100 epochs of 250 iters = 25k iters total.
# This fits comfortably in 16GB. Adjust --epochs up for more quality, down for speed.
# Checkpoints save to /kaggle/working/deep_sr/ every 2000 iters.

# !python train_deep_sr.py --device cuda --epochs 100 --batch 8 --patch 32 --unfolding-iters 5 --iters-per-epoch 250 --save-every 2000 --log-every 50 --workers 2

# Quick test run (1 epoch = 250 iters, ~10 min on T4) — uncomment to sanity-check first:
!python train_deep_sr.py --device cuda --epochs 1 --batch 8 --patch 32 --iters-per-epoch 250 --save-every 500 --log-every 25 --workers 2

In [ ]:
# Cell 6 — Run full training after the quick test above looks good.
# Uncomment and run when ready. ~24-48h on T4 for 100 epochs.

# !python train_deep_sr.py --device cuda --epochs 100 --batch 8 --patch 32 \
#     --iters-per-epoch 250 --save-every 2000 --log-every 50 --workers 2

In [ ]:
# Cell 7 — List outputs + download instructions
import os
out_dir = '/kaggle/working/deep_sr'
if os.path.isdir(out_dir):
    print('Trained checkpoints available for download:')
    for f in sorted(os.listdir(out_dir)):
        p = os.path.join(out_dir, f)
        size_mb = os.path.getsize(p) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')
    print('\n→ Download these from the "Output" panel on the right.')
    print('→ Drop best_checkpoint.pth into artifacts/deep_sr/ in your local repo.')
    print('→ inference.py will auto-load it on next run.')
else:
    print('No checkpoints yet. Run training cells above first.')

## After training
1. Download `best_checkpoint.pth` from the notebook's `Output` panel (right sidebar).
2. In your local VectoVecto repo: `mkdir -p artifacts/deep_sr && cp best_checkpoint.pth artifacts/deep_sr/`
3. Run inference: `python inference.py` — it auto-detects the DRUNet checkpoint and uses it (4× scale, 5 unfolding iterations, 8-way TTA).

## Tips
- **Quicker checkpoint**: change `--epochs 1` to `--epochs 5` for a 30-min checkpoint you can test locally.
- **Better quality**: bump `--epochs` to 200 or 400 and `--iters-per-epoch` to 500 for 100k+ iters — Real-ESRGAN paper used ~200k.
- **Watch the GAN**: if `D loss` → 0 while `G loss` keeps climbing, the discriminator won — lower `--lr-d` to `5e-5` and restart.
- **Save often**: Kaggle kernels can disconnect after ~12h. Set `--save-every 1000` so you keep progress.
- **Resume**: not implemented yet. If disconnected, restart from latest checkpoint (would need a `--resume` flag — easy to add).